In [0]:
%pip install polars

In [0]:
%run "../lib/00_common"

In [0]:
# =============================================
# Silver: limpieza de strings en olist_orders
# =============================================

import polars as pl
from pathlib import Path

# Leer Parquet desde Bronze
orders_path = f"{BRONZE_ROOT}/orders/olist_orders_dataset.parquet"
df_orders = pl.read_parquet(orders_path)

print("Schema original:")
print(df_orders.schema)
print(f"\nFilas: {len(df_orders)}")

# Identificar columnas string automáticamente y aplicar trim
string_cols = [col for col, dtype in df_orders.schema.items() if dtype == pl.String]

print(f"Columnas string encontradas: {string_cols}\n")

df_orders_clean = df_orders.with_columns([
    pl.col(col).str.strip_chars().alias(col)
    for col in string_cols
])

print(" Trim aplicado a todas las columnas string.")

In [0]:
# ============================================================
# Silver: columnas derivadas en olist_orders
# ============================================================

df_orders_clean = df_orders_clean.with_columns([
    
    # delivery_days: días entre compra y entrega al cliente
    (
        (pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp"))
        .dt.total_days()
    ).alias("delivery_days"),

    # is_late: True si llegó después de la fecha estimada
    (
        pl.col("order_delivered_customer_date") > pl.col("order_estimated_delivery_date")
    ).alias("is_late"),

])

print(" Columnas derivadas agregadas.\n")

display(df_orders_clean.head().to_pandas())

In [0]:
# ============================================================
# Silver: seleccionar columnas finales y guardar
# ============================================================

df_orders_silver = df_orders_clean.select([
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "is_late",
    "_run_date",
])

# Guardar en Silver
import os
silver_orders_path = f"{SILVER_ROOT}/orders/olist_orders_silver.parquet"
os.makedirs(os.path.dirname(silver_orders_path), exist_ok=True)
df_orders_silver.write_parquet(silver_orders_path)

print(f"   Orders Silver guardado.")
print(f"   Filas:    {len(df_orders_silver):,}")
print(f"   Columnas: {len(df_orders_silver.columns)}")
print(f"   Ruta:     {silver_orders_path}")

In [0]:
# ============================================================
# Silver Orders_items: trim strings + total_price
# ============================================================

order_items_path = f"{BRONZE_ROOT}/order_items/olist_order_items_dataset.parquet"
# → "/Volumes/workspace/elt_lab/elt_volume/olist_project/bronze/order_items/olist_order_items_dataset.parquet"

# Leer todo para no perder columnas al guardar en Silver
df_order_items = pl.read_parquet(order_items_path)

# Trim a columnas string
string_cols = [col for col, dtype in df_order_items.schema.items() if dtype == pl.String]

df_order_items_clean = df_order_items.with_columns([
    pl.col(col).str.strip_chars().alias(col)
    for col in string_cols
])

# Calcular total_price
df_order_items_clean = df_order_items_clean.with_columns([
    (pl.col("price") + pl.col("freight_value")).alias("total_price")
])

print(" total_price calculado.\n")

# Vista previa
display(df_order_items_clean.head().to_pandas())

In [0]:
# ============================================================
# Silver: seleccionar columnas finales y guardar
# ============================================================

df_order_items_silver = df_order_items_clean.select([
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value",
    "total_price",
    "_run_date",
])

# Guardar en Silver
silver_order_items_path = f"{SILVER_ROOT}/order_items/olist_order_items_silver.parquet"
os.makedirs(os.path.dirname(silver_order_items_path), exist_ok=True)
df_order_items_silver.write_parquet(silver_order_items_path)

print(f"   Order Items Silver guardado.")
print(f"   Filas:    {len(df_order_items_silver):,}")
print(f"   Columnas: {len(df_order_items_silver.columns)}")
print(f"   Ruta:     {silver_order_items_path}")

In [0]:
# ============================================================
# Silver: products
# ============================================================
import polars as pl

# Leer desde Bronze
products_path    = f"{BRONZE_ROOT}/products/olist_products_dataset.parquet"
category_path    = f"{BRONZE_ROOT}/product_category/product_category_name_translation.parquet"

df_products      = pl.read_parquet(products_path)
df_translation   = pl.read_parquet(category_path).select([
    "product_category_name",
    "product_category_name_english"
])

print(f"Filas antes de limpieza: {len(df_products):,}")

In [0]:
# ============================================================
# Silver: products
# ============================================================
string_cols = [col for col, dtype in df_products.schema.items() if dtype == pl.String]
df_products_clean = df_products.with_columns([
    pl.col(col).str.strip_chars().alias(col)
    for col in string_cols
])

# Eliminar filas sin product_category_name
df_products_clean = df_products_clean.filter(
    pl.col("product_category_name").is_not_null() & 
    (pl.col("product_category_name") != "")
)

print(f"Filas después de eliminar sin categoría: {len(df_products_clean):,}")

# Join con traducción (inner para quedarnos solo con los que tienen traducción al inglés)
df_products_clean = df_products_clean.join(
    df_translation,
    on="product_category_name",
    how="inner"  # inner: elimina los que no tienen traducción
)

# Reemplazar columna portugués con la traducción en inglés
df_products_clean = df_products_clean.drop("product_category_name").rename({
    "product_category_name_english": "product_category_name"
})

print(f"Filas finales: {len(df_products_clean):,}")
print(f"\nVista previa:")
# print(df_products_clean.select(["product_id", "product_category_name"]).head(10))

print(df_products_clean)

In [0]:
# ============================================================
# Silver: seleccionar columnas finales y guardar products
# ============================================================

df_products_silver = df_products_clean.select([
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "_run_date",
])

# Guardar en Silver
silver_products_path = f"{SILVER_ROOT}/products/olist_products_silver.parquet"
os.makedirs(os.path.dirname(silver_products_path), exist_ok=True)
df_products_silver.write_parquet(silver_products_path)

print(f"✅ Products Silver guardado.")
print(f"   Filas:    {len(df_products_silver):,}")
print(f"   Columnas: {len(df_products_silver.columns)}")
print(f"   Ruta:     {silver_products_path}")

In [0]:
# ============================================================
# Silver: olist_customers_dataset
# ============================================================

# Leer desde Bronze
customers_path = f"{BRONZE_ROOT}/customers/olist_customers_dataset.parquet"
df_customers = pl.read_parquet(customers_path)

# Trim general a todos los strings
string_cols = [col for col, dtype in df_customers.schema.items() if dtype == pl.String]
df_customers_clean = df_customers.with_columns([
    pl.col(col).str.strip_chars().alias(col)
    for col in string_cols
])

# Estandarizar city y state
df_customers_clean = df_customers_clean.with_columns([
    pl.col("customer_city").str.to_lowercase().alias("customer_city"),
    pl.col("customer_state").str.to_uppercase().alias("customer_state"),
])

print(df_customers_clean.select(["customer_city", "customer_state"]).head(10))

In [0]:
# ============================================================
# Silver: seleccionar columnas finales y guardar customers
# ============================================================

df_customers_silver = df_customers_clean.select([
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "_run_date",
])

# Guardar en Silver
silver_customers_path = f"{SILVER_ROOT}/customers/olist_customers_silver.parquet"
os.makedirs(os.path.dirname(silver_customers_path), exist_ok=True)
df_customers_silver.write_parquet(silver_customers_path)

print(f"   Customers Silver guardado.")
print(f"   Filas:    {len(df_customers_silver):,}")
print(f"   Columnas: {len(df_customers_silver.columns)}")
print(f"   Ruta:     {silver_customers_path}")

In [0]:
# ============================================================
# Silver: olist_sellers_dataset
# ============================================================
import polars as pl

# Leer desde Bronze
sellers_path = f"{BRONZE_ROOT}/sellers/olist_sellers_dataset.parquet"
df_sellers = pl.read_parquet(sellers_path)

print(f"Filas: {len(df_sellers):,}")

# Trim general a todos los strings
string_cols = [col for col, dtype in df_sellers.schema.items() if dtype == pl.String]
df_sellers_clean = df_sellers.with_columns([
    pl.col(col).str.strip_chars().alias(col)
    for col in string_cols
])

# Estandarizar city y state
df_sellers_clean = df_sellers_clean.with_columns([
    pl.col("seller_city").str.to_lowercase().alias("seller_city"),
    pl.col("seller_state").str.to_uppercase().alias("seller_state"),
])

print(f" Estandarización completada.\n")
print(df_sellers_clean.select(["seller_city", "seller_state"]).head(10))

In [0]:
# ============================================================
# Silver: seleccionar columnas finales y guardar sellers
# ============================================================

df_sellers_silver = df_sellers_clean.select([
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "_run_date",
])

# Guardar en Silver
silver_sellers_path = f"{SILVER_ROOT}/sellers/olist_sellers_silver.parquet"
os.makedirs(os.path.dirname(silver_sellers_path), exist_ok=True)
df_sellers_silver.write_parquet(silver_sellers_path)

print(f"   Sellers Silver guardado.")
print(f"   Filas:    {len(df_sellers_silver):,}")
print(f"   Columnas: {len(df_sellers_silver.columns)}")
print(f"   Ruta:     {silver_sellers_path}")

In [0]:
# ============================================================
# Silver: order_payments
# ============================================================
import polars as pl

# Leer desde Bronze
payments_path = f"{BRONZE_ROOT}/order_payments/olist_order_payments_dataset.parquet"
df_payments = pl.read_parquet(payments_path)

# Trim general a todos los strings
string_cols = [col for col, dtype in df_payments.schema.items() if dtype == pl.String]
df_payments_clean = df_payments.with_columns([
    pl.col(col).str.strip_chars().alias(col)
    for col in string_cols
])

# Calcular total_paid_per_order
total_paid = df_payments_clean.group_by("order_id").agg(
    pl.col("payment_value").sum().alias("total_paid_per_order")
)

# Unir de vuelta al dataframe original
df_payments_clean = df_payments_clean.join(total_paid, on="order_id", how="left")

print("total_paid_per_order calculado.\n")
print(df_payments_clean)
# print(df_payments_clean.select([
#     "order_id",
#     "payment_type",
#     "payment_value",
#     "total_paid_per_order"
# ]).head(10))

In [0]:
# ============================================================
#  Silver: seleccionar columnas finales y guardar payments
# ============================================================

df_payments_silver = df_payments_clean.select([
    "order_id",
    "payment_type",
    "payment_value",
    "total_paid_per_order",
    "_run_date",
])

# Guardar en Silver
silver_payments_path = f"{SILVER_ROOT}/order_payments/olist_order_payments_silver.parquet"
os.makedirs(os.path.dirname(silver_payments_path), exist_ok=True)
df_payments_silver.write_parquet(silver_payments_path)

print(f"   Payments Silver guardado.")
print(f"   Filas:    {len(df_payments_silver):,}")
print(f"   Columnas: {len(df_payments_silver.columns)}")
print(f"   Ruta:     {silver_payments_path}")

In [0]:
# ============================================================
# Silver: geolocation
# ============================================================
import polars as pl

# Leer desde Bronze
geolocation_path = f"{BRONZE_ROOT}/geolocation/olist_geolocation_dataset.parquet"
df_geolocation = pl.read_parquet(geolocation_path)

print(f"Filas antes de limpieza: {len(df_geolocation):,}")

# Trim general a todos los strings
string_cols = [col for col, dtype in df_geolocation.schema.items() if dtype == pl.String]
df_geolocation_clean = df_geolocation.with_columns([
    pl.col(col).str.strip_chars().alias(col)
    for col in string_cols
])

# Estandarizar city y state
df_geolocation_clean = df_geolocation_clean.with_columns([
    pl.col("geolocation_city").str.to_lowercase().alias("geolocation_city"),
    pl.col("geolocation_state").str.to_uppercase().alias("geolocation_state"),
])

# Filtrar coordenadas fuera de Brasil
df_geolocation_clean = df_geolocation_clean.filter(
    (pl.col("geolocation_lat").is_between(-33, 5)) &
    (pl.col("geolocation_lng").is_between(-74, -35))
)

print(f"Filas después de filtrar coordenadas: {len(df_geolocation_clean):,}")

# Eliminar duplicados — quedarse con el primer registro por zip code
df_geolocation_clean = df_geolocation_clean.unique(
    subset=["geolocation_zip_code_prefix"],
    keep="first"
)

print(f"Filas después de eliminar duplicados: {len(df_geolocation_clean):,}")

# Verificación
print(df_geolocation_clean.select([
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state"
]).head(10))

In [0]:
# ============================================================
# Silver: seleccionar columnas finales y guardar geolocation
# ============================================================

df_geolocation_silver = df_geolocation_clean.select([
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
    "_run_date",
])

# Guardar en Silver
silver_geolocation_path = f"{SILVER_ROOT}/geolocation/olist_geolocation_silver.parquet"
os.makedirs(os.path.dirname(silver_geolocation_path), exist_ok=True)
df_geolocation_silver.write_parquet(silver_geolocation_path)

print(f"   Geolocation Silver guardado.")
print(f"   Filas:    {len(df_geolocation_silver):,}")
print(f"   Columnas: {len(df_geolocation_silver.columns)}")
print(f"   Ruta:     {silver_geolocation_path}")